In [ ]:
import re
from pathlib import Path

import pandas as pd

In [70]:
for path in Path("data").rglob("*"):
    if ".csv" in path.suffix.lower():
        print(path)

data/extended/Constraint_min_volume.csv
data/extended/Constraint_min_flow.csv
data/kernel/Historical_day_ahead_price_2015_2025.csv
data/kernel/Historical_volume_2015_2024.csv
data/kernel/Historical_inflow_1958_2025.csv
data/kernel/Unit_commitment_decisions.csv
data/kernel/Synthetic_water_value_2015_2024.csv


In [71]:
DATA_DIR = Path("data")

paths = {
    "price": DATA_DIR / "kernel/Historical_day_ahead_price_2015_2025.csv",
    "volume": DATA_DIR / "kernel/Historical_volume_2015_2024.csv",
    "inflow": DATA_DIR / "kernel/Historical_inflow_1958_2025.csv",
    "target": DATA_DIR / "kernel/Unit_commitment_decisions.csv",
    "water_value": DATA_DIR / "kernel/Synthetic_water_value_2015_2024.csv",
    "min_volume": DATA_DIR / "extended/Constraint_min_volume.csv",
    "min_flow": DATA_DIR / "extended/Constraint_min_flow.csv",
}

raw = {}

for name, path in paths.items():
    raw[name] = pd.read_csv(path)

In [ ]:
price = raw["price"].copy()
price["date"] = pd.to_datetime(price["date"], utc=True, format="mixed")
price = price.sort_values("date").set_index("date")
price = price.rename(columns={"Day-ahead price (EUR/MWh)": "price"})

steps = price.index.to_series().diff()
switch = price.index[steps == pd.Timedelta("15min")][0]
print("15-minute steps start at:", switch)

# Hourly grid covering the whole range
hourly_idx = pd.date_range(
    price.index.min().floor("h"), price.index.max().floor("h"), freq="h"
)

# take the quarter starting at the top of each hour -> forward-fill within the hour
price_h = (
    price["price"]
    .reindex(hourly_idx.union(price.index))
    .sort_index()
    .ffill(limit=4)
    .reindex(hourly_idx)
)

# sanity: how does 'top of hour' compare to hourly mean over the 4 quarters?
price_q = price["price"]
price_mean_h = price_q.resample("1h").mean()
cmp = pd.DataFrame({"top_of_hour": price_h, "mean4": price_mean_h}).dropna()
print("corr(top, mean):", cmp.corr().iloc[0, 1].round(6))
print("mean abs diff (EUR/MWh):", (cmp.top_of_hour - cmp.mean4).abs().mean().round(4))
print(cmp.loc[switch : switch + pd.Timedelta(days=7)].head(10))

15-minute steps start at: 2025-09-30 22:15:00+00:00
corr(top, mean): 0.999886
mean abs diff (EUR/MWh): 0.0636
                           top_of_hour    mean4
2025-09-30 23:00:00+00:00        65.59  64.3350
2025-10-01 00:00:00+00:00        63.66  63.5200
2025-10-01 01:00:00+00:00        63.78  63.7525
2025-10-01 02:00:00+00:00        63.56  63.8350
2025-10-01 03:00:00+00:00        63.09  66.2350
2025-10-01 04:00:00+00:00        68.34  69.4675
2025-10-01 05:00:00+00:00        78.03  81.3125
2025-10-01 06:00:00+00:00        97.70  83.6575
2025-10-01 07:00:00+00:00        85.00  75.0250
2025-10-01 08:00:00+00:00        74.33  69.6675


In [ ]:
t_min = price_h.index.min()
t_max = max(
    price_h.index.max(),
    pd.to_datetime(raw["inflow"]["date"], utc=True, format="mixed").max(),
    pd.to_datetime(raw["target"]["starttime"], utc=True, format="mixed").max()
    + pd.Timedelta(hours=167),
)
timeline = pd.date_range(t_min, t_max, freq="h")


def to_hourly_index(df, freq_expected):
    ts = pd.to_datetime(df["date"], utc=True, format="mixed")
    assert ts.is_unique and ts.is_monotonic_increasing or ts.is_unique
    return ts


coverage = pd.DataFrame(index=timeline)
coverage["price"] = coverage.index.isin(price_h.index)
inflow_ts = pd.to_datetime(raw["inflow"]["date"], utc=True, format="mixed")
coverage["inflow"] = coverage.index.isin(inflow_ts)

t_start = pd.to_datetime(raw["target"]["starttime"], utc=True, format="mixed")
# each case covers [starttime, starttime + 167h]
case_mask = np.zeros(len(timeline), bool)
starts = t_start.sort_values()
pos = starts.index if False else None
# vectorized coverage: count of active cases per hour
active = pd.Series(0, index=timeline)
for s in starts:
    active.loc[s : s + pd.Timedelta(hours=167)] += 1
coverage["in_case"] = active.values > 0

print("hours total:", len(timeline))
print(coverage.sum())
print("\nFirst/last case coverage check:")
print("cases start:", starts.min(), "end:", starts.max() + pd.Timedelta(hours=167))

# Where do cases lack price or inflow coverage?
gaps = coverage[coverage.in_case & ~(coverage.price & coverage.inflow)]
print("case-hours missing price or inflow:", len(gaps))
if len(gaps):
    print(gaps.index.min(), "->", gaps.index.max())

hours total: 96432
price      96432
inflow     94568
in_case    70272
dtype: int64

First/last case coverage check:
cases start: 2015-01-01 00:00:00+00:00 end: 2023-01-06 23:00:00+00:00
case-hours missing price or inflow: 0


In [ ]:
# Daily tables -> lookup by calendar day (as-of, forward-fill within the case only)
vol = raw["volume"].copy()
vol["date"] = pd.to_datetime(vol["date"], utc=True, format="mixed")
wv = raw["water_value"].copy()
wv["date"] = pd.to_datetime(wv["date"], utc=True, format="mixed")
inflow = raw["inflow"].copy()
inflow["date"] = pd.to_datetime(inflow["date"], utc=True, format="mixed")

# long inflow, hourly
inf_long = inflow.melt("date", var_name="reservoir", value_name="inflow")

# per (case, hour) grid
cases = raw["target"][["Run No", "starttime"]].copy()
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, format="mixed")

rows = []
for _, case in cases.iterrows():
    hours = pd.date_range(case.starttime, periods=168, freq="h")
    df = pd.DataFrame(
        {
            "Run No": case["Run No"],
            "timestamp": hours,
            "t": np.arange(168),
            "date": hours.floor("D"),
        }
    )
    rows.append(df)
grid = pd.concat(rows, ignore_index=True)  # 2922 * 168 = 490,896 rows

# join hourly price and hourly inflow on timestamp; daily volume/water value on date
grid = grid.merge(
    price_h.rename("price"), left_on="timestamp", right_index=True, how="left"
)
grid = (
    grid.merge(
        inf_long,
        left_on=[
            "timestamp",
        ],
        right_on=["date_y"] if False else None,
        how="left",
    )
    if False
    else grid
)  # inflow join in next step (pivoted wide is easier)
inf_wide = inflow.set_index("date")
grid = grid.join(
    inf_wide, on="timestamp"
)  # 18 inflow columns (incl. r_Vest_Vassdraget)
grid = grid.join(vol.set_index("date").add_prefix("vol_"), on="date")
grid = grid.join(wv.set_index("date").add_prefix("wv_"), on="date")

print(grid.shape)
print("missing price:", grid["price"].isna().sum())
print("missing volume cells:", grid.filter(like="vol_").isna().sum().sum())

(490896, 57)
missing price: 0
missing volume cells: 0


In [75]:
# ---- 1. per-unit reservoir mapping (simplified, 1 reservoir per plant) ----
PLANT_RESERVOIR = {
    "Tokke": "Bandak",
    "Vinje": "Vinjevatn",
    "Hogga": "Vinjevatn",
    "Byrte": "Byrtevatn",
    "Songa": "Songavatn",
    "Kjela": "Totak",
    "Vesle Kjela": "Kjelavatn",
    "Lio": "Vaamarvatn",
    "Haukeli": "Foersvatn",
}  # verify against the topology PDF before trusting

# per-unit case-level features: initial volume and end water value of the unit's reservoir
cases = raw["target"][["Run No", "starttime"]].copy()
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, format="mixed")
vol_ix = raw["volume"].set_index("date")  # daily, date col already parsed
wv_ix = raw["water_value"].set_index("date")
for res in PLANT_RESERVOIR.values():
    pass  # (columns picked per-unit below, so no wide pivoting needed)

case_feats = cases.copy()


def _case_boundary(row):
    res = PLANT_RESERVOIR[row_plant_of[row["Run No"]]]
    ...


# simpler: build per-unit table directly

# ---- 2. attach reservoir to each label row ----
labels["reservoir"] = labels["plant"].map(PLANT_RESERVOIR)
assert labels["reservoir"].notna().all(), "unmapped plant!"

# ---- 3. case-level boundary features per unit ----
wv_cols = raw["water_value"].set_index("date")
vol_cols = raw["volume"].set_index("date")


def case_level_feats(row):
    res = row["reservoir"]
    start = row["starttime"]
    end_plus = start + pd.Timedelta(days=8)
    return pd.Series(
        {
            "init_vol_res": vol_cols.at[start.floor("D"), res],
            "end_wv_res": wv_cols.at[end_plus.floor("D"), res],
        }
    )


case_level = labels.groupby(["Run No", "unit_id"]).head(1)  # one row per case-unit
case_level = pd.concat(
    [case_level, case_level.apply(case_level_feats, axis=1)], axis=1
)[["Run No", "unit_id", "init_vol_res", "end_wv_res"]]

# ---- 4. hourly grid features (from your existing grid cell) ----
# grid: (Run No, t, timestamp, date, price, <18 inflow cols>, vol_*, wv_*)
grid2 = grid.copy()
grid2["hour"] = grid2.timestamp.dt.hour
grid2["dow"] = grid2.timestamp.dt.dayofweek
grid2["month"] = grid2.timestamp.dt.month

# ---- 5. merge everything into the master table ----
labels_g = (
    labels.merge(grid2, on=["Run No", "t"], how="left").merge(  # hourly features
        case_level, on=["Run No", "unit_id"], how="left"
    )  # case-level features
)
# per-row unit-specific columns
labels_g["vol_res"] = labels_g.apply(lambda r: r[f"vol_{r.reservoir}"], axis=1)
labels_g["wv_res"] = labels_g.apply(lambda r: r[f"wv_{r.reservoir}"], axis=1)
labels_g["inflow_res"] = labels_g.apply(lambda r: r[f"{r.reservoir}"], axis=1)
labels_g["margin"] = labels_g["price"] - labels_g["wv_res"]

assert labels_g["price"].isna().sum() == 0
assert labels_g["init_vol_res"].isna().sum() == 0
print(labels_g.shape)  # expect (490896, ~40)

TypeError: can only concatenate str (not "Timedelta") to str

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

FEATS_NUM = [
    "price",
    "margin",
    "t",
    "hour",
    "dow",
    "month",
    "vol_res",
    "wv_res",
    "inflow_res",
    "init_vol_res",
    "end_wv_res",
    "up_inflow_sum",
    "up_wv_min",
]

df = labels_g.copy()  # your merged long table with engineered columns
df["hour"] = df.timestamp.dt.hour
df["dow"] = df.timestamp.dt.dayofweek
df["month"] = df.timestamp.dt.month
df["unit_id"] = df["unit_id"].astype("category")

split_date = pd.Timestamp("2022-01-01", tz="UTC")
train = df[df.starttime < split_date]
val = df[df.starttime >= split_date]

clf = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=50,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    early_stopping_rounds=100,
)
clf.fit(
    train[FEATS_NUM + ["unit_id"]],
    train.on,
    eval_set=[(val[FEATS_NUM + ["unit_id"]], val.on)],
    verbose=200,
)

val_pred = clf.predict_proba(val[FEATS_NUM + ["unit_id"]])[:, 1]
print("pooled AUC:", roc_auc_score(val.on, val_pred))
print("per-unit AUC:")
for uid, g in val.groupby("unit_id"):
    print(f"  {uid:16s} {roc_auc_score(g.on, val_pred[g.index]):.4f}")

NameError: name 'labels_g' is not defined

In [ ]:
target = pd.read_csv(paths["target"])

print(target.head())
print(target.columns.tolist())

   Run No   starttime  total_calculation_time  result_committed_Hogga_G1_t0  \
0       1  2015-01-01               48.950692                             1   
1       2  2015-01-02               56.048899                             1   
2       3  2015-01-03               44.280307                             1   
3       4  2015-01-04               40.848453                             1   
4       5  2015-01-05               44.036279                             1   

   result_committed_Hogga_G1_t1  result_committed_Hogga_G1_t2  \
0                             1                             1   
1                             1                             1   
2                             1                             1   
3                             1                             1   
4                             1                             1   

   result_committed_Hogga_G1_t3  result_committed_Hogga_G1_t4  \
0                             1                             1   
1   

In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[^a-zA-Z0-9]+", "_", str(column).strip().lower()).strip("_")
        for column in df.columns
    ]
    return df


data = {name: normalize_columns(df) for name, df in raw.items()}

feature_tables = []
for name, df in data.items():
    if name == "target":
        continue

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # Keep date unchanged, prefix all other columns
    df = df.rename(
        columns={
            column: f"{name}_{column}" for column in df.columns if column != "date"
        }
    )

    feature_tables.append(df)
features = feature_tables[0]

for df in feature_tables[1:]:
    features = features.merge(df, on="date", how="outer")

features = features.sort_values("date")
features

,date,price_day_ahead_price_eur_mwh,volume_kjelavatn,volume_staavatn,volume_langesae,volume_foersvatn,volume_venemo,volume_botnedalsvatn,volume_bitdalsvatn,volume_vaamarvatn,...,water_value_vatjern,min_volume_totak,min_volume_byrtevatn,min_volume_staavatn,min_flow_b_kjelavatn,min_flow_b_hyljelihyl,min_flow_b_vinjevatn,min_flow_b_byrtevatn,min_flow_b_bandak,min_flow_r_vest_vassdraget
0,1958-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1958-01-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1958-01-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1958-01-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1958-01-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602713,2025-12-31 22:45:00+00:00,65.31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
602714,2025-12-31 23:00:00+00:00,65.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.3,0.5,1.0,0.0,4.0,5.0
602715,2025-12-31 23:15:00+00:00,64.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
602716,2025-12-31 23:30:00+00:00,63.31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
target = data["target"].copy()

print(target.shape)
print(target.columns.tolist())
print(target.head())

(2922, 2355)
['run_no', 'starttime', 'total_calculation_time', 'result_committed_hogga_g1_t0', 'result_committed_hogga_g1_t1', 'result_committed_hogga_g1_t2', 'result_committed_hogga_g1_t3', 'result_committed_hogga_g1_t4', 'result_committed_hogga_g1_t5', 'result_committed_hogga_g1_t6', 'result_committed_hogga_g1_t7', 'result_committed_hogga_g1_t8', 'result_committed_hogga_g1_t9', 'result_committed_hogga_g1_t10', 'result_committed_hogga_g1_t11', 'result_committed_hogga_g1_t12', 'result_committed_hogga_g1_t13', 'result_committed_hogga_g1_t14', 'result_committed_hogga_g1_t15', 'result_committed_hogga_g1_t16', 'result_committed_hogga_g1_t17', 'result_committed_hogga_g1_t18', 'result_committed_hogga_g1_t19', 'result_committed_hogga_g1_t20', 'result_committed_hogga_g1_t21', 'result_committed_hogga_g1_t22', 'result_committed_hogga_g1_t23', 'result_committed_hogga_g1_t24', 'result_committed_hogga_g1_t25', 'result_committed_hogga_g1_t26', 'result_committed_hogga_g1_t27', 'result_committed_hogga

In [ ]:
training_data = target.merge(features, on="date", how="left")

KeyError: 'date'